# PGMPy Probabilistic Inference API

A guided exploration of pgmpy's core probabilistic inference abstractions:
- Building Bayesian networks
- Inference engines (VariableElimination, BeliefPropagation)
- Query methods (observational inference, MAP queries)
- Evidence-based conditioning


## Imports and Setup


In [2]:
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Install pgmpy if needed
# import subprocess
# import importlib.util

# if importlib.util.find_spec('pgmpy') is None:
#     subprocess.check_call(
#         [sys.executable, '-m', 'pip', 'install', 'pgmpy', '-q']
#     )

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
import pandas as pd
import numpy as np

# TODO(ai_gp): Use standard initialization
_LOG = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')

# Part 1: Library Overview

## What Problem Does PGMPy Solve?

PGMPy enables probabilistic reasoning over graphical models:
- **Graphical Models**: Represent complex probability distributions as graphs where nodes are variables and edges encode dependencies
- **Bayesian Networks**: Directed acyclic graphs (DAGs) representing causal and statistical relationships
- **Probabilistic Inference**: Answer probability questions given observed evidence: "What is P(X=x | Y=y)?"
- **MAP Queries**: Find most probable variable assignments given evidence

## Key Abstractions

1. **BayesianNetwork**: The graph structure (nodes and edges)
2. **TabularCPD**: Conditional probability distributions (quantify relationships)
3. **InferenceEngine** (VariableElimination, BeliefPropagation): Compute probabilities given evidence
4. **Query Results**: Distributions over variables conditioned on evidence

## How the Pieces Fit Together

# TODO(ai_gp): Use a graphviz graph
```
BayesianNetwork (structure)
    + TabularCPD (quantification)
    ↓
InferenceEngine (instantiate with model)
    ↓
query(variables, evidence) → Distribution
map_query(variables, evidence) → Most probable assignment
```

# Part 2: Core Primitives

## Primitive 1: BayesianNetwork

Represents the graph structure (directed acyclic graph).

### Minimal Construction


In [8]:
# Smallest possible Bayesian network: 2 nodes with 1 edge
model = DiscreteBayesianNetwork([('A', 'B')])

print(f"Type: {type(model)}")
print(f"Nodes: {model.nodes()}")
print(f"Edges: {model.edges()}")

Type: <class 'pgmpy.models.DiscreteBayesianNetwork.DiscreteBayesianNetwork'>
Nodes: ['A', 'B']
Edges: [('A', 'B')]


### Inspect the Object


In [9]:
# View important attributes
print(f"Has CPDs: {model.get_cpds()}")
print(f"Check valid: {model.check_model()}")

# BayesianNetwork is invalid without CPDs
try:
    model.check_model()
    print("Model is valid")
except Exception as e:
    print(f"Model is INVALID: {type(e).__name__}")
    print(f"  Reason: Missing CPDs for nodes {model.nodes()}")

Has CPDs: []


ValueError: No CPD associated with A

### Important Methods


In [ ]:
# Exploring the network structure
model = BayesianNetwork([('A', 'B'), ('A', 'C'), ('B', 'C')])

print(f"Parents of B: {model.get_parents('B')}")
print(f"Children of A: {model.get_children('A')}")
print(f"All predecessors of C: {model.get_predecessors('C')}")

### Mental Model

A **BayesianNetwork** is:
- A directed acyclic graph (DAG) encoding independence assumptions
- An edge A → B means "A probabilistically influences B"
- Each node requires a Conditional Probability Distribution (CPD)
- Nodes without parents are root nodes (prior distributions)
- Nodes with parents are conditional on those parents

---

## Primitive 2: TabularCPD (Conditional Probability Distribution)

Quantifies the relationship between variables as a probability table.

### Minimal Construction


In [ ]:
# CPD for a node with no parents (prior)
cpd_a = TabularCPD(
    variable='A',
    variable_card=2,  # A can take 2 values: 0 or 1
    values=[[0.6], [0.4]]  # P(A=0)=0.6, P(A=1)=0.4
)

print(f"Type: {type(cpd_a)}")
print(cpd_a)

### CPD with Parents


In [ ]:
# CPD for B given A (B has parent A)
# B's distribution depends on A's value
cpd_b = TabularCPD(
    variable='B',
    variable_card=2,
    values=[
        [0.9, 0.2],  # P(B=0 | A=0)=0.9, P(B=0 | A=1)=0.2
        [0.1, 0.8]   # P(B=1 | A=0)=0.1, P(B=1 | A=1)=0.8
    ],
    evidence=['A'],  # B depends on A
    evidence_card=[2]  # A has 2 values
)

print(cpd_b)
print(f"\nRead as: When A=0, P(B=0)=0.9 and P(B=1)=0.1")
print(f"         When A=1, P(B=0)=0.2 and P(B=1)=0.8")

### Important Methods


In [ ]:
# Inspect a CPD
print(f"Variable: {cpd_b.variable}")
print(f"Cardinality (num values): {cpd_b.cardinality}")
print(f"Evidence: {cpd_b.variables[1:]}")
print(f"Evidence cardinality: {cpd_b.cardinality[1:]}")

# Access probability values
print(f"\nP(B=0 | A=1): {cpd_b.values[0, 1]}")
print(f"P(B=1 | A=0): {cpd_b.values[1, 0]}")

### Mental Model

A **TabularCPD** is:
- A probability table organizing conditional distributions
- For a node with k parents (each with c_i values), table has ∏c_i columns
- Each column represents one evidence configuration
- Each column sums to 1.0 (valid probability distribution)
- Columns are ordered by evidence cardinality (rightmost parent varies fastest)

---

## Primitive 3: InferenceEngine (VariableElimination)

Computes posterior probabilities given evidence.

### Minimal Construction and Setup


In [ ]:
# Create a complete model
model = BayesianNetwork([('A', 'B')])

# Add CPDs
cpd_a = TabularCPD('A', 2, [[0.6], [0.4]])
cpd_b = TabularCPD(
    'B', 2,
    [[0.9, 0.2], [0.1, 0.8]],
    evidence=['A'], evidence_card=[2]
)

model.add_cpds(cpd_a, cpd_b)
model.check_model()

# Create inference engine
inference = VariableElimination(model)

print(f"Type: {type(inference)}")
print(f"Model: {inference.model.nodes()}")

### Important Methods


In [ ]:
# Query: compute P(B | A=1)
result = inference.query(variables=['B'], evidence={'A': 1})

print(f"Type of result: {type(result)}")
print(result)
print(f"\nP(B=0 | A=1) = {result.values[0]}")
print(f"P(B=1 | A=1) = {result.values[1]}")

### Mental Model

An **InferenceEngine** is:
- An algorithm that computes posterior probabilities from a graphical model
- Takes a fitted BayesianNetwork (with CPDs)
- Answers conditional probability questions: P(query_vars | evidence)
- VariableElimination: Exact, efficient for many practical models
- BeliefPropagation: Exact, optimized for repeated queries

---

## Primitive 4: Query Results

The output of inference queries: probability distributions.


In [ ]:
# Query result is a Factor object
result = inference.query(variables=['B'], evidence={'A': 0})

print(f"Type: {type(result)}")
print(f"Variables: {result.variables}")
print(f"Cardinality: {result.cardinality}")
print(f"\nAs array: {result.values}")
print(f"\nAs DataFrame:")
result_df = result.to_dataframe()
print(result_df)

# Part 3: Composition Examples

## Example 1: Minimal End-to-End Workflow

Rain → Sprinkler → Grass Wet


In [ ]:
# 1. Define structure
model = BayesianNetwork([('Rain', 'GrassWet'), ('Sprinkler', 'GrassWet')])

# 2. Define CPDs
cpd_rain = TabularCPD('Rain', 2, [[0.8], [0.2]])  # Usually no rain
cpd_sprinkler = TabularCPD('Sprinkler', 2, [[0.6], [0.4]])  # Usually off

# 3. Grass wet depends on both Rain and Sprinkler
cpd_grass = TabularCPD(
    'GrassWet', 2,
    [
        [0.99, 0.1, 0.1, 0.01],  # P(Grass=dry | ...)
        [0.01, 0.9, 0.9, 0.99]   # P(Grass=wet | ...)
    ],
    evidence=['Rain', 'Sprinkler'],
    evidence_card=[2, 2]
)

model.add_cpds(cpd_rain, cpd_sprinkler, cpd_grass)
model.check_model()

print("Model is valid!")
print(f"Nodes: {model.nodes()}")
print(f"Edges: {model.edges()}")

In [ ]:
# 4. Create inference engine and query
inference = VariableElimination(model)

# Query 1: Grass is wet, what's probability of rain?
result = inference.query(variables=['Rain'], evidence={'GrassWet': 1})
print("P(Rain | Grass=wet):")
print(result)

# Query 2: Both rain and sprinkler, what's probability grass is wet?
result = inference.query(
    variables=['GrassWet'],
    evidence={'Rain': 1, 'Sprinkler': 1}
)
print("\nP(Grass | Rain=yes, Sprinkler=yes):")
print(result)

## Example 2: Add Multi-State Variables

Temperature (cold, warm, hot) → Comfort (bad, good)


In [ ]:
# Variables can have more than 2 states
model = BayesianNetwork([('Temperature', 'Comfort')])

# Temperature: 0=cold, 1=warm, 2=hot
cpd_temp = TabularCPD(
    'Temperature', 3,
    [[0.3], [0.5], [0.2]]  # 30% cold, 50% warm, 20% hot
)

# Comfort: 0=bad, 1=good
# Changes based on temperature
cpd_comfort = TabularCPD(
    'Comfort', 2,
    [
        [0.8, 0.1, 0.6],  # P(Comfort=bad | ...)
        [0.2, 0.9, 0.4]   # P(Comfort=good | ...)
    ],
    evidence=['Temperature'],
    evidence_card=[3]
)

model.add_cpds(cpd_temp, cpd_comfort)
model.check_model()

inference = VariableElimination(model)
result = inference.query(variables=['Temperature'], evidence={'Comfort': 1})

print("P(Temperature | Comfort=good):")
for i, val in enumerate(result.values):
    print(f"  P(Temp={i}) = {val:.3f}")

## Example 3: Chain Structure

X → Y → Z (three-node chain)


In [ ]:
model = BayesianNetwork([('X', 'Y'), ('Y', 'Z')])

cpd_x = TabularCPD('X', 2, [[0.4], [0.6]])
cpd_y = TabularCPD(
    'Y', 2,
    [[0.9, 0.3], [0.1, 0.7]],
    evidence=['X'], evidence_card=[2]
)
cpd_z = TabularCPD(
    'Z', 2,
    [[0.8, 0.2], [0.2, 0.8]],
    evidence=['Y'], evidence_card=[2]
)

model.add_cpds(cpd_x, cpd_y, cpd_z)
model.check_model()

inference = VariableElimination(model)

# Query with evidence at the beginning
result = inference.query(variables=['Z'], evidence={'X': 1})
print("P(Z | X=1):")
print(result)

# Query with evidence in the middle
result = inference.query(variables=['X'], evidence={'Z': 1})
print("\nP(X | Z=1) [backwards inference]:")
print(result)

## Example 4: V-Structure (Collider)

A ← B → C (classic collider pattern)


In [ ]:
# V-structure: both A and C point to B
model = BayesianNetwork([('A', 'B'), ('C', 'B')])

cpd_a = TabularCPD('A', 2, [[0.5], [0.5]])
cpd_c = TabularCPD('C', 2, [[0.5], [0.5]])

# B depends on both A and C
cpd_b = TabularCPD(
    'B', 2,
    [
        [0.99, 0.6, 0.6, 0.05],  # P(B=0 | ...)
        [0.01, 0.4, 0.4, 0.95]   # P(B=1 | ...)
    ],
    evidence=['A', 'C'],
    evidence_card=[2, 2]
)

model.add_cpds(cpd_a, cpd_c, cpd_b)
model.check_model()

inference = VariableElimination(model)

# Without evidence on B, A and C are independent
# With evidence on B, they become dependent (explaining away)
result_before = inference.query(variables=['A'], evidence={'C': 1})
result_after = inference.query(variables=['A'], evidence={'B': 1, 'C': 1})

print("P(A | C=1) [without B]:")
print(result_before.values)
print("\nP(A | C=1, B=1) [with B]:")
print(result_after.values)
print("\nNote: Probabilities change when B is observed (explaining away effect)")

# Part 4: API Patterns

## Pattern 1: Model Construction and Validation


In [ ]:
# Pattern: define structure → add CPDs → validate → create engine

# Step 1: Structure
model = BayesianNetwork([('A', 'B'), ('B', 'C')])

# Step 2: Add CPDs
cpd_a = TabularCPD('A', 2, [[0.3], [0.7]])
cpd_b = TabularCPD(
    'B', 2,
    [[0.8, 0.2], [0.2, 0.8]],
    evidence=['A'], evidence_card=[2]
)
cpd_c = TabularCPD(
    'C', 2,
    [[0.9, 0.1], [0.1, 0.9]],
    evidence=['B'], evidence_card=[2]
)
model.add_cpds(cpd_a, cpd_b, cpd_c)

# Step 3: Validation
is_valid = model.check_model()
print(f"Model valid: {is_valid}")

# Step 4: Create inference engine
inference = VariableElimination(model)

## Pattern 2: Query Pattern

Basic query: `inference.query(variables, evidence)`


In [ ]:
# Pattern: query(variables=[...], evidence={...})
# Returns posterior distribution P(variables | evidence)

# Single variable, no evidence
result = inference.query(variables=['A'])
print("P(A): prior distribution")
print(result)

# Single variable with evidence
result = inference.query(variables=['C'], evidence={'A': 1})
print("\nP(C | A=1): posterior distribution")
print(result)

# Multiple variables with evidence
result = inference.query(variables=['A', 'B'], evidence={'C': 0})
print("\nP(A, B | C=0): joint posterior")
print(result)

## Pattern 3: MAP Query Pattern

Find the most probable assignment.


In [ ]:
# Pattern: map_query(variables, evidence)
# Returns most probable variable assignment given evidence

# Most probable value of C given A=1
result = inference.map_query(variables=['C'], evidence={'A': 1})
print(f"MAP(C | A=1): {result}")
print(f"Interpretation: Most likely value of C is {result['C']}")

# Most probable joint assignment
result = inference.map_query(variables=['B', 'C'], evidence={'A': 0})
print(f"\nMAP(B, C | A=0): {result}")
print(f"Interpretation: Most likely (B, C) = ({result['B']}, {result['C']})")

## Pattern 4: Inference Engine Alternatives


In [ ]:
# VariableElimination: Default, efficient for many models
inference_ve = VariableElimination(model)
result_ve = inference_ve.query(variables=['C'], evidence={'A': 1})

# BeliefPropagation: Optimized for repeated queries
# Note: Works best on singly-connected networks
try:
    inference_bp = BeliefPropagation(model)
    result_bp = inference_bp.query(variables=['C'], evidence={'A': 1})
    print("Both engines produce the same result:")
    print(f"VariableElimination: {result_ve.values}")
    print(f"BeliefPropagation:   {result_bp.values}")
except Exception as e:
    print(f"BeliefPropagation note: {e}")

## Pattern 5: Evidence Specification


In [ ]:
# Evidence is specified as {variable: value}
# Can use multiple pieces of evidence

# Single observation
result = inference.query(variables=['A'], evidence={'C': 1})
print("P(A | C=1)")

# Multiple observations
result = inference.query(
    variables=['A'],
    evidence={'B': 1, 'C': 0}
)
print("\nP(A | B=1, C=0)")
print(result)

# Querying variables can also have evidence
result = inference.query(
    variables=['A', 'B'],
    evidence={'C': 1}
)
print("\nP(A, B | C=1) - joint distribution of A and B")
print(result)

# Part 5: Object Relationships

Understanding how objects interact:


In [ ]:
import pandas as pd

# Create example objects
bn = BayesianNetwork([('A', 'B'), ('A', 'C')])
cpd_a = TabularCPD('A', 2, [[0.3], [0.7]])
cpd_b = TabularCPD(
    'B', 2,
    [[0.8, 0.2], [0.2, 0.8]],
    evidence=['A'], evidence_card=[2]
)
cpd_c = TabularCPD(
    'C', 2,
    [[0.6, 0.4], [0.4, 0.6]],
    evidence=['A'], evidence_card=[2]
)
bn.add_cpds(cpd_a, cpd_b, cpd_c)

inference = VariableElimination(bn)
query_result = inference.query(variables=['B'], evidence={'A': 1})

# Display relationships
relationships = pd.DataFrame([
    {
        'Object': 'BayesianNetwork',
        'Created By': 'Direct instantiation',
        'Consumed By': 'InferenceEngine',
        'Purpose': 'Encode graph structure and dependencies'
    },
    {
        'Object': 'TabularCPD',
        'Created By': 'Direct instantiation',
        'Consumed By': 'BayesianNetwork.add_cpds()',
        'Purpose': 'Quantify conditional probability relationships'
    },
    {
        'Object': 'InferenceEngine',
        'Created By': 'VariableElimination(model) or BeliefPropagation(model)',
        'Consumed By': 'query(), map_query() methods',
        'Purpose': 'Compute posterior probabilities from model'
    },
    {
        'Object': 'Query Result',
        'Created By': 'inference.query() or inference.map_query()',
        'Consumed By': 'User code for decision-making',
        'Purpose': 'Answer probability questions given evidence'
    },
])

print(relationships.to_string(index=False))

# Part 6: Interactive Exploration

## Experiment 1: How Does Evidence Change Posteriors?


In [ ]:
# Simple model: Cause → Effect
model = BayesianNetwork([('Cause', 'Effect')])
cpd_cause = TabularCPD('Cause', 2, [[0.5], [0.5]])  # Cause is equally likely
cpd_effect = TabularCPD(
    'Effect', 2,
    [[0.95, 0.1], [0.05, 0.9]],  # Effect strongly correlated with Cause
    evidence=['Cause'], evidence_card=[2]
)
model.add_cpds(cpd_cause, cpd_effect)
model.check_model()

inference = VariableElimination(model)

# Prior on Cause (no evidence)
prior = inference.query(variables=['Cause'])
print("Prior P(Cause):")
print(f"  P(Cause=0) = {prior.values[0]:.3f}")
print(f"  P(Cause=1) = {prior.values[1]:.3f}")

# Posterior given Effect=1 (strong evidence for Cause)
posterior = inference.query(variables=['Cause'], evidence={'Effect': 1})
print("\nPosterior P(Cause | Effect=1):")
print(f"  P(Cause=0) = {posterior.values[0]:.3f}")
print(f"  P(Cause=1) = {posterior.values[1]:.3f}")

print("\nObservation: Effect=1 makes Cause=1 much more likely!")
print(f"Belief shift: {prior.values[1]:.3f} → {posterior.values[1]:.3f}")

## Experiment 2: Why Variable Cardinality Matters


In [ ]:
# Create two models: one with 2 states, one with 5 states

# Binary model
model_binary = BayesianNetwork([('Weather', 'Traffic')])
cpd_w = TabularCPD('Weather', 2, [[0.7], [0.3]])
cpd_t = TabularCPD(
    'Traffic', 2,
    [[0.95, 0.3], [0.05, 0.7]],
    evidence=['Weather'], evidence_card=[2]
)
model_binary.add_cpds(cpd_w, cpd_t)
model_binary.check_model()

inf_binary = VariableElimination(model_binary)
result = inf_binary.query(variables=['Weather'], evidence={'Traffic': 1})
print("Binary model (Weather: clear/rainy):")
print(f"  CPD table size: 2x2")
print(result)

# Multi-state model
model_multi = BayesianNetwork([('Weather', 'Traffic')])
cpd_w = TabularCPD('Weather', 5, [[0.5], [0.2], [0.15], [0.1], [0.05]])
cpd_t = TabularCPD(
    'Traffic', 3,
    [
        [0.9, 0.4, 0.2, 0.1, 0.05],  # P(T=light)
        [0.08, 0.5, 0.5, 0.4, 0.3],  # P(T=moderate)
        [0.02, 0.1, 0.3, 0.5, 0.65]  # P(T=heavy)
    ],
    evidence=['Weather'], evidence_card=[5]
)
model_multi.add_cpds(cpd_w, cpd_t)
model_multi.check_model()

inf_multi = VariableElimination(model_multi)
result = inf_multi.query(variables=['Weather'], evidence={'Traffic': 2})
print("\nMulti-state model (Weather: 5 states, Traffic: 3 states):")
print(f"  CPD table size: 3x5")
print(result)

## Experiment 3: Computational Complexity


In [ ]:
import time

# Create a chain of increasing length
# X₀ → X₁ → X₂ → ... → X_n

for chain_length in [3, 5, 7, 10]:
    # Build chain
    edges = [(f'X{i}', f'X{i+1}') for i in range(chain_length)]
    model = BayesianNetwork(edges)
    
    # Add CPDs
    for i in range(chain_length):
        if i == 0:
            cpd = TabularCPD(f'X{i}', 2, [[0.5], [0.5]])
        else:
            cpd = TabularCPD(
                f'X{i}', 2,
                [[0.9, 0.2], [0.1, 0.8]],
                evidence=[f'X{i-1}'], evidence_card=[2]
            )
        model.add_cpds(cpd)
    
    model.check_model()
    inference = VariableElimination(model)
    
    # Time a query
    start = time.time()
    result = inference.query(variables=[f'X{chain_length-1}'], evidence={'X0': 1})
    elapsed = time.time() - start
    
    print(f"Chain length {chain_length}: {elapsed*1000:.2f}ms")

# Part 7: Cheat Sheet

## Core Objects Quick Reference


In [ ]:
cheat_sheet = pd.DataFrame([
    {
        'Object': 'BayesianNetwork',
        'Purpose': 'Graph structure',
        'Construction': 'BayesianNetwork([(a, b), (b, c)])',
        'Key Method': 'add_cpds(), check_model()'
    },
    {
        'Object': 'TabularCPD',
        'Purpose': 'Quantify relationships',
        'Construction': 'TabularCPD(variable, cardinality, values, evidence, evidence_card)',
        'Key Method': '.values, .cardinality'
    },
    {
        'Object': 'VariableElimination',
        'Purpose': 'Exact inference (default)',
        'Construction': 'VariableElimination(model)',
        'Key Method': 'query(), map_query()'
    },
    {
        'Object': 'Query Result',
        'Purpose': 'Posterior distribution',
        'Construction': 'inference.query(...)',
        'Key Method': '.values, .to_dataframe()'
    },
])

print(cheat_sheet.to_string(index=False))

## Most Useful Methods


In [ ]:
methods = pd.DataFrame([
    {
        'Method': 'inference.query(variables, evidence)',
        'Returns': 'Posterior distribution P(variables | evidence)',
        'Example': 'inference.query(["B"], {"A": 1})'
    },
    {
        'Method': 'inference.map_query(variables, evidence)',
        'Returns': 'Most likely assignment (dict)',
        'Example': 'inference.map_query(["B"], {"A": 1})'
    },
    {
        'Method': 'model.add_cpds(*cpds)',
        'Returns': 'None',
        'Example': 'model.add_cpds(cpd_a, cpd_b)'
    },
    {
        'Method': 'model.check_model()',
        'Returns': 'True if valid, raises exception otherwise',
        'Example': 'model.check_model()'
    },
    {
        'Method': 'result.to_dataframe()',
        'Returns': 'Pandas DataFrame with probability values',
        'Example': 'result.to_dataframe()'
    },
])

print(methods.to_string(index=False))

## Typical Workflow (End-to-End)


In [ ]:
# Complete, minimal end-to-end example

# 1. Define the graph structure
model = BayesianNetwork([('Disease', 'Test')])

# 2. Define probability distributions
cpd_disease = TabularCPD('Disease', 2, [[0.99], [0.01]])
cpd_test = TabularCPD(
    'Test', 2,
    [[0.95, 0.1], [0.05, 0.9]],
    evidence=['Disease'], evidence_card=[2]
)

# 3. Add CPDs to model
model.add_cpds(cpd_disease, cpd_test)
model.check_model()

# 4. Create inference engine
inference = VariableElimination(model)

# 5. Answer probability questions
# Q: Someone tests positive. How likely do they have the disease?
posterior = inference.query(variables=['Disease'], evidence={'Test': 1})

print("P(Disease | Test=positive):")
print(f"  P(Disease=0) = {posterior.values[0]:.4f}")
print(f"  P(Disease=1) = {posterior.values[1]:.4f}")
print(f"\nInterpretation: Even with positive test, only {posterior.values[1]:.1%} likely to have disease")
print("(This is base rate fallacy - prior was very low)")

## Summary: The Mental Model

- **BayesianNetwork**: "Here's the causal/dependency structure"
- **TabularCPD**: "Here's how strong each relationship is"
- **InferenceEngine**: "Use this structure + relationships to answer probability questions"
- **query()**: "What's the distribution over X given we observed Y=y?"
- **map_query()**: "What's the single most likely X given we observed Y=y?"

The API is remarkably consistent across different inference algorithms.
Switch from VariableElimination to BeliefPropagation by just changing one line of instantiation.
